# BTC Regime Prediction -- XGBoost Walk-Forward Classifier

## Overview

This notebook is **Phase 2** of the BTC regime classifier. The Gaussian HMM in notebook 1
assigned regime labels to every historical day. Here we train a **supervised XGBoost
classifier** to predict tomorrow's regime from features observable today.

**Pipeline per walk-forward fold:**
1. Split data into expanding train / validation / test windows
2. Fit `RobustScaler` on training data only
3. **Optuna TPE** (100 trials, multivariate) searches over 9 XGBoost hyperparameters
4. Best params retrained on train + val combined before scoring test
5. Test set scored: per-class probabilities + hard regime prediction

**Walk-forward design:** Expanding train window, fixed 90-day val and test windows,
advancing 90 days per fold. 18 folds covering 2021-2026.

**Features (32 total):**
- 32 price-derived: returns, volatility, momentum, drawdown (4 timescales: 20d/60d/120d/252d), recovery from trough (3 timescales: 20d/60d/120d), volume, RSI, Bollinger, ATR, regime state

**Key finding:** `dd_60d` (60-day drawdown from rolling high) is by far the dominant
feature. The full drawdown family (20d, 60d, 120d, 252d) covers corrections through macro
bear cycles at every timescale.

**Hyperparameters searched by Optuna:**

| Parameter | Range | Controls |
|-----------|--------|----------|
| `n_estimators` | 100-1000 | Number of boosting trees |
| `max_depth` | 3-7 | Maximum tree depth |
| `learning_rate` | 0.001-0.15 (log) | Gradient step size |
| `subsample` | 0.5-1.0 | Fraction of rows per tree |
| `colsample_bytree` | 0.4-1.0 | Fraction of features per tree |
| `min_child_weight` | 1-20 | Minimum leaf sample weight |
| `gamma` | 0-2 | Minimum gain to split |
| `reg_alpha` | 0-3 | L1 regularisation |
| `reg_lambda` | 0.1-5 | L2 regularisation |

**Output:** `data/predictions/btc_regime_predictions.parquet`

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from xgboost import XGBClassifier
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (
    accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
)
import shap

REPO      = Path('../../..').resolve()
CACHE_DIR = REPO / 'live_trading' / 'cache' / 'daily'
LABELS_DIR = REPO / 'topics' / 'regime-classifier' / 'data' / 'labels'
PREDS_DIR  = REPO / 'topics' / 'regime-classifier' / 'data' / 'predictions'
PREDS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE  = 42
REGIME_NAMES  = ['Bull', 'Recovery/Chop', 'Bear', 'Extreme Bear', 'Extreme Bull']
REGIME_COLORS = ['#1565c0', '#2e7d32', '#f57c00', '#d32f2f', '#6a1b9a']

print('Repo:', REPO)
print('optuna:', optuna.__version__)
print('shap  :', shap.__version__)

---
## 1. Load Data

Two files are loaded and aligned to the same date index:

1. **`btc_regimes.parquet`** (from notebook 1) -- provides the `regime` Viterbi labels used as the training target (`next_regime`), and the `regime_online` forward-filter labels used as a feature.

2. **`BTCUSDT_daily.parquet`** -- BTC OHLCV for constructing the 28 technical features. Aligned to the regime label index so features and targets share the same date axis.

The last available date in the regime labels determines the OOS prediction coverage. Run notebook 1 first if the labels are stale.

In [ ]:
# HMM regime labels
regime_df = pd.read_parquet(LABELS_DIR / 'btc_regimes.parquet')
print(f'Regime labels : {regime_df.index[0].date()} to {regime_df.index[-1].date()}  ({len(regime_df)} rows)')
print('Regime distribution:')
print(regime_df['regime'].value_counts().sort_index().to_string())

# BTC OHLCV — align to regime index
btc = pd.read_parquet(CACHE_DIR / 'BTCUSDT_daily.parquet')
btc.index = pd.to_datetime(btc.index).tz_localize(None)
btc = btc.sort_index().reindex(regime_df.index)

print(f'\nBTC OHLCV aligned : {btc.index[0].date()} to {btc.index[-1].date()}')
btc[['Close', 'Volume']].tail(3)

---
## 2. Feature Engineering

32 features are constructed from BTC OHLCV data.
Every feature is computed from data available at the current day's close — no lookahead.
The **target** is `next_regime`: tomorrow's Viterbi regime label, shifted back by 1 day.

| Group | Features | Why useful |
|-------|----------|------------|
| Lagged returns | `ret_1d` to `ret_20d` | Recent price direction and momentum |
| Rolling volatility | `rvol_5d`, `10d`, `20d`, `60d` | Current volatility level |
| Vol ratio | `rvol_ratio_5_20` | Whether vol is expanding or contracting |
| Momentum | `mom_5d`, `10d`, `20d`, `60d` | Cumulative return over window |
| Momentum spread | `mom_spread_5_20` | Divergence between short and long momentum |
| Drawdown | `dd_20d`, `dd_60d`, `dd_120d`, `dd_252d` | Distance from recent peak at 4 timescales — top feature family |
| Recovery | `recov_20d`, `recov_60d`, `recov_120d` | Distance from recent trough — model knows when price is bouncing even if still deep in drawdown |
| Volume | `vol_chg_5d`, `vol_zscore` | Volume anomalies often precede regime shifts |
| RSI | `rsi_14` | Overbought / oversold positioning |
| Bollinger | `bb_width`, `bb_pos` | Volatility compression and price location |
| ATR | `atr_14` | Normalised daily trading range |
| Regime state | `regime`, `regime_lag1`, `regime_duration` | Today's perceived regime and persistence |

**Why `dd_252d`?** The 60-day drawdown is by far the dominant feature (gain importance ~0.17).
It captures corrections well but cannot distinguish a mid-bull correction (deep short-term,
shallow 1-year) from a macro bear (deep at every timescale). The 252-day drawdown adds this
macro dimension.

**Why the recovery features?** `dd_60d` stays deep for weeks after a crash bottoms while
price is already recovering. The recovery features (`recov_*d`) measure how far price has
bounced from its rolling low within each window — near 0 at the crash bottom, rising quickly
during recovery. This gives the model explicit signal to exit Bear predictions earlier at
regime turns.

In [ ]:
close  = btc['Close']
high   = btc['High']
low    = btc['Low']
volume = btc['Volume']

log_ret = np.log(close / close.shift(1))

feat = pd.DataFrame(index=close.index)

# Lagged returns
for lag in [1, 2, 3, 5, 10, 20]:
    feat[f'ret_{lag}d'] = log_ret.shift(lag)

# Rolling volatility
for win in [5, 10, 20, 60]:
    feat[f'rvol_{win}d'] = log_ret.rolling(win).std()

# Volatility regime indicator
feat['rvol_ratio_5_20'] = feat['rvol_5d'] / feat['rvol_20d'].replace(0, np.nan)

# Momentum (cumulative return)
for win in [5, 10, 20, 60]:
    feat[f'mom_{win}d'] = np.log(close / close.shift(win))

# Momentum spread
feat['mom_spread_5_20'] = feat['mom_5d'] - feat['mom_20d']

# Drawdown from rolling highs (20d, 60d, 120d, 252d)
for win in [20, 60, 120, 252]:
    roll_max = close.rolling(win).max()
    feat[f'dd_{win}d'] = (close - roll_max) / roll_max

# Recovery from rolling lows (20d, 60d, 120d)
# Complement to drawdown: near 0 at crash bottom, rising during recovery
# even when dd_*d is still deep. Helps model exit Bear predictions earlier.
for win in [20, 60, 120]:
    roll_min = close.rolling(win).min()
    feat[f'recov_{win}d'] = (close - roll_min) / roll_min.replace(0, np.nan)

# Volume features
log_vol  = np.log(volume.clip(lower=1))
vol_sma  = log_vol.rolling(20).mean()
vol_std  = log_vol.rolling(20).std().replace(0, np.nan)
feat['vol_chg_5d']  = log_vol - log_vol.shift(5)
feat['vol_zscore']  = (log_vol - vol_sma) / vol_std

# RSI-14
delta    = close.diff()
avg_gain = delta.clip(lower=0).ewm(com=13, adjust=False).mean()
avg_loss = (-delta).clip(lower=0).ewm(com=13, adjust=False).mean()
rs = avg_gain / avg_loss.replace(0, np.nan)
feat['rsi_14'] = 100 - (100 / (1 + rs))

# Bollinger band features
sma_20 = close.rolling(20).mean()
std_20 = close.rolling(20).std().replace(0, np.nan)
feat['bb_width'] = (2 * std_20) / sma_20
feat['bb_pos']   = (close - sma_20) / std_20

# Normalised ATR
tr = pd.concat([
    high - low,
    (high - close.shift(1)).abs(),
    (low  - close.shift(1)).abs(),
], axis=1).max(axis=1)
feat['atr_14'] = tr.rolling(14).mean() / close.replace(0, np.nan)

# Regime features
feat['regime']      = regime_df['regime_online']
feat['regime_lag1'] = regime_df['regime_online'].shift(1)
feat['regime_duration'] = (
    feat.groupby((feat['regime'] != feat['regime'].shift()).cumsum())
    .cumcount() + 1
)

# ── Target & finalise ─────────────────────────────────────────────────────────
feat['next_regime'] = regime_df['regime'].shift(-1)

feat = feat.dropna()
n_classes = int(feat['next_regime'].max()) + 1
FEAT_COLS = [c for c in feat.columns if c not in ('next_regime',)]

print(f'Features  : {len(FEAT_COLS)}')
print(f'Rows      : {len(feat)}')
print(f'Date range: {feat.index[0].date()} to {feat.index[-1].date()}')
print(f'Classes   : {n_classes}')
feat[FEAT_COLS[:6]].describe().round(4)


---
## 3. Walk-Forward Configuration

The walk-forward splits the labelled history into a sequence of folds. Each fold has three non-overlapping windows:

```
Full history:
|<-- train (expands) -->|<- val (90d) ->|<- test (90d) ->|   fold 0
                        |<-- train (expands) -->|<- val ->|<- test ->|   fold 1
                                                ...
```

**Train window expands** -- each new fold adds 90 days of data to training. This mirrors live deployment: as time passes, more historical data becomes available to retrain the model.

**Val window** is fixed at 90 days. Optuna's 100 trials maximise accuracy on this window only. The val window is *inside* the known history -- the model can be tuned against it.

**Test window** is the truly OOS window. It is never seen during hyperparameter search. After Optuna selects the best params on val, the final model is **retrained on train + val combined** before scoring test. This maximises the data available to the model that will score the OOS window.

**Why retrain on train + val before scoring test?** The val set was used to select hyperparameters, making it partially in-sample for that purpose. Before scoring the genuinely unseen test window, the final model should have as much data as possible -- so train + val are combined for the final fit. This is standard walk-forward practice.

**Configuration parameters:**

| Parameter | Value | Rationale |
|-----------|-------|----------|
| `min_train_days` | 270 | Minimum rows before first fold -- ensures at least one full regime cycle in training |
| `val_days` | 90 | 3 months -- enough data to meaningfully evaluate accuracy |
| `test_days` | 90 | 3 months of OOS predictions per fold |
| `step_days` | 90 | Fold advances 90 days -- non-overlapping OOS windows, no double-counting |
| `n_optuna_trials` | 100 | Minimum for 9-parameter multivariate TPE to converge |

With ~1,980 labelled days and `min_train_days` = 270, the first fold starts after 270 + 90 = 360 days of warm-up. Approximately 18 folds are generated, covering 18 x 90 = 1,620 OOS days.

In [ ]:
WF_CONFIG = {
    'min_train_days':  270,   # minimum train before first fold (expands after)
    'val_days':         90,   # 3 months — Optuna validates here
    'test_days':        90,   # 3 months — OOS scoring
    'step_days':        90,   # advance 3 months per fold
    'n_optuna_trials': 100,   # trials per fold (increase for deeper search)
}

# Estimate fold count
labelled = feat[feat['next_regime'].notna()]
n_dates  = len(labelled.index.unique())
tr_d, va_d, te_d, step_d = (
    WF_CONFIG['min_train_days'], WF_CONFIG['val_days'],
    WF_CONFIG['test_days'],  WF_CONFIG['step_days'],
)
n_folds_est = max(0, (n_dates - tr_d - va_d - te_d) // step_d + 1)

print(f'Labelled dates       : {n_dates}')
print(f'Min window required  : {tr_d + va_d + te_d}')
print(f'Estimated folds      : ~{n_folds_est}')
print(f'Optuna trials/fold   : {WF_CONFIG["n_optuna_trials"]}')


---
## 4. Optuna Objective and Fold Runner

### The Optuna TPE Sampler

**TPE (Tree-structured Parzen Estimator)** is a Bayesian optimisation method. Unlike random search (which explores uniformly) or grid search (which scales exponentially), TPE builds a probabilistic model of the objective function:

1. It splits all completed trials into 'good' (top 25% by val accuracy) and 'bad' (bottom 75%)
2. It fits a kernel density estimator $l(x)$ to the good configurations and $g(x)$ to the bad ones
3. New trial parameters are sampled by maximising the ratio $l(x) / g(x)$ -- concentrating search on regions that produced good results

This means TPE gets progressively smarter with each trial, while random search stays equally dumb throughout.

### Why `multivariate=True`?

Standard TPE models each hyperparameter independently, missing interactions. With `multivariate=True`, Optuna fits the joint density over all 9 parameters simultaneously. This matters because `n_estimators` and `learning_rate` interact strongly: a high learning rate (e.g. 0.1) converges fast and needs fewer trees (e.g. 200). A low learning rate (e.g. 0.005) needs many more trees (e.g. 800) to achieve the same fit. A univariate sampler treating these independently would recommend combinations like low learning rate + few trees, which are suboptimal.

### Why 100 Trials?

TPE needs enough trials to build a reliable density estimate. With 9 parameters and multivariate modelling, roughly the first 25 trials are essentially random exploration (not enough data for the density estimator yet). The remaining 75 trials benefit from the learned structure. Fewer than 100 trials would mean the search exits before the TPE model has converged to the good region.

### Why Accuracy as the Objective?

The regime predictions are used as a binary gate in downstream strategies (trade or don't trade). What matters is that the hard predicted label is correct -- not the probability distribution. Accuracy directly measures this. Alternative objectives:
- **Balanced accuracy:** Better for imbalanced classes, but `compute_sample_weight('balanced')` already corrects for imbalance during training. Plain accuracy is simpler and more interpretable for a gate.
- **Log-loss:** Would optimise probability calibration, not classification boundary. Useful if probability thresholding is the primary use case, but the hard label is most commonly used.

### Class Weighting

The four regime classes are not perfectly balanced (~28% each in-sample, but some folds may be unbalanced). Without weighting, XGBoost minimises total error, which means it would sacrifice accuracy on rare classes to maintain accuracy on common ones. `compute_sample_weight('balanced')` assigns higher weight to under-represented classes in each fold, ensuring the model learns all four regimes equally.

### Per-Fold Steps (summary)

1. Fill any missing feature values with the **training set median** (not global median -- prevents leakage)
2. Fit `RobustScaler` on train only; transform val and test
3. Run 100 Optuna trials; each trial trains on scaled train, evaluates on scaled val
4. Combine train + val; refit scaler; train final model with best params
5. Score test: store probabilities, hard prediction, true label, fold index, val accuracy

In [ ]:
from sklearn.utils.class_weight import compute_sample_weight

def _run_fold(train_df, val_df, test_df, feat_cols, n_classes, config, fold_idx):
    train_median = train_df[feat_cols].median()

    def _xy(df):
        X = df[feat_cols].fillna(train_median).values
        y = df['next_regime'].values.astype(int)
        return X, y

    X_tr, y_tr = _xy(train_df)
    X_va, y_va = _xy(val_df)
    X_te, y_te = _xy(test_df)

    scaler = RobustScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_va_s = scaler.transform(X_va)
    X_te_s = scaler.transform(X_te)

    sw_tr = compute_sample_weight('balanced', y_tr)

    def _objective(trial):
        params = {
            'n_estimators':     trial.suggest_int('n_estimators', 100, 1000),
            'max_depth':        trial.suggest_int('max_depth', 3, 7),
            'learning_rate':    trial.suggest_float('learning_rate', 0.001, 0.15, log=True),
            'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
            'gamma':            trial.suggest_float('gamma', 0.0, 2.0),
            'reg_alpha':        trial.suggest_float('reg_alpha', 0.0, 3.0),
            'reg_lambda':       trial.suggest_float('reg_lambda', 0.1, 5.0),
            'objective':        'multi:softprob',
            'num_class':        n_classes,
            'eval_metric':      'mlogloss',
            'verbosity':        0,
            'random_state':     RANDOM_STATE,
        }
        model = XGBClassifier(**params)
        model.fit(X_tr_s, y_tr, sample_weight=sw_tr)
        return accuracy_score(y_va, model.predict(X_va_s))

    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE + fold_idx, multivariate=True),
    )
    study.optimize(_objective, n_trials=config['n_optuna_trials'], show_progress_bar=False)

    best_p = dict(study.best_params)
    best_p.update({'objective': 'multi:softprob', 'num_class': n_classes,
                   'eval_metric': 'mlogloss', 'verbosity': 0, 'random_state': RANDOM_STATE})

    X_tv  = np.vstack([X_tr_s, X_va_s])
    y_tv  = np.concatenate([y_tr, y_va])
    sw_tv = compute_sample_weight('balanced', y_tv)

    final_model = XGBClassifier(**best_p)
    final_model.fit(X_tv, y_tv, sample_weight=sw_tv)

    probs_te = final_model.predict_proba(X_te_s)
    preds_te = final_model.predict(X_te_s)

    result_df = pd.DataFrame(
        {f'p_regime_{i}': probs_te[:, i] for i in range(n_classes)},
        index=test_df.index,
    )
    result_df['pred_regime'] = preds_te
    result_df['true_regime'] = y_te
    result_df['fold']        = fold_idx
    result_df['val_acc']     = study.best_value

    meta = {
        'fold': fold_idx,
        'train_start': train_df.index[0].date(), 'train_end': train_df.index[-1].date(),
        'val_start':   val_df.index[0].date(),   'val_end':   val_df.index[-1].date(),
        'test_start':  test_df.index[0].date(),  'test_end':  test_df.index[-1].date(),
        'val_acc': study.best_value, 'best_params': best_p,
        'best_trial': study.best_trial.number,
        'n_train': len(train_df), 'n_val': len(val_df), 'n_test': len(test_df),
        'model': final_model, 'scaler': scaler, 'feat_cols': feat_cols,
    }
    return result_df, meta


print('_run_fold defined (class weighting, no early stopping).')


---
## 5. Run Walk-Forward

Executes all folds sequentially. Per-fold output shows:
- **Train and test date ranges** -- confirms the expanding window is working correctly
- **val_acc** -- Optuna's best validation accuracy for this fold. Varies by fold: transition periods (late 2021, mid 2022) are inherently harder to classify and produce lower val_acc. This is expected and healthy.
- **n_est** -- the selected number of estimators. A consistently low value (e.g. 150) across folds suggests the learning rate may be too high. A consistently high value (e.g. 900) suggests the learning rate is too low.
- **depth** -- selected max_depth. Consistent depth = 3 suggests the problem is mostly linear in the features. Depth = 7 suggests complex interactions are important.

**Runtime:** Approximately 15-25 minutes for all 18 folds with 100 trials each. The exact time depends on the winning `n_estimators` value -- folds where Optuna selects 800+ trees take longer to retrain on train+val.

In [ ]:
def run_walk_forward(feat_df, n_classes, config):
    data  = feat_df[feat_df['next_regime'].notna()].sort_index()
    dates = data.index.unique().sort_values()
    n     = len(dates)

    tr_d   = config.get('min_train_days', config.get('train_days', 270))
    va_d   = config['val_days']
    te_d   = config['test_days']
    step_d = config['step_days']

    all_results, all_meta = [], []
    fold_idx = 0
    start    = tr_d + va_d

    while start + te_d <= n:
        tr_dates = dates[0 : start - va_d]
        va_dates = dates[start - va_d         : start       ]
        te_dates = dates[start                : start + te_d]

        train_df = data[data.index.isin(tr_dates)]
        val_df   = data[data.index.isin(va_dates)]
        test_df  = data[data.index.isin(te_dates)]

        if len(train_df) < 100 or len(val_df) < 20 or len(test_df) < 10:
            start += step_d
            continue

        print(
            f'  Fold {fold_idx:02d}  '
            f'train {tr_dates[0].date()} to {tr_dates[-1].date()}  '
            f'test {te_dates[0].date()} to {te_dates[-1].date()}',
            end='  '
        )

        result_df, meta = _run_fold(
            train_df, val_df, test_df, FEAT_COLS, n_classes, config, fold_idx
        )
        all_results.append(result_df)
        all_meta.append(meta)

        print(
            f'val_acc={meta["val_acc"]:.3f}  '
            f'n_est={meta["best_params"]["n_estimators"]}  '
            f'depth={meta["best_params"]["max_depth"]}'
        )

        fold_idx += 1
        start    += step_d

    if not all_results:
        raise RuntimeError('No folds completed — check data length vs config.')

    results_df = pd.concat(all_results).sort_index()
    print(f'\nDone. {fold_idx} folds  |  {len(results_df)} test rows')
    return results_df, all_meta


print('Running walk-forward')

results_df, all_meta = run_walk_forward(feat, n_classes, WF_CONFIG)



---
## 6. Out-of-Sample Accuracy

The OOS accuracy is computed across all 1,620 test days from all 18 folds -- this is the headline number that characterises the classifier's predictive power.

**Interpreting the numbers:**

- **OOS accuracy**: the fraction of test days where `pred_regime == true_regime`. Target: above 0.80.
- **Naive baseline**: always predicting the most frequent regime (Recovery/Chop, ~30% of days). This represents zero predictive information -- the lift over baseline measures what the model actually contributes.
- **Lift over baseline**: OOS accuracy - baseline accuracy. A model with 85% accuracy vs a 30% baseline has a +55% lift. This is the real signal.

**Per-regime accuracy** reveals the asymmetry:
- **Bull and Bear** are easiest to classify: strong directional momentum and clearly distinct vol/drawdown signatures
- **Recovery/Chop** is in the middle: average features, adjacent to both Bull and Bear
- **Extreme Bear** is hardest: rare (only ~15% of days in the OOS period), and the transition into it is brief, limiting training examples. Misclassifying an Extreme Bear day as regular Bear is a conservative error -- the strategy would still avoid it.

**What 85% accuracy means in practice:** The model is wrong ~1 in 6 days. Most errors are adjacent-regime errors (predicting Bull when it's Chop, or Bear when it's Extreme Bear) rather than distant errors (predicting Bull during an Extreme Bear crash). The confusion matrix in the next section confirms this.

In [ ]:
oos_acc = accuracy_score(results_df['true_regime'], results_df['pred_regime'])

# Naive baseline: always predict most frequent regime
most_freq     = int(results_df['true_regime'].mode().iloc[0])
baseline_acc  = accuracy_score(
    results_df['true_regime'],
    np.full(len(results_df), most_freq, dtype=int)
)

print(f'OOS accuracy         : {oos_acc:.3f}')
print(f'Naive baseline (R{most_freq})  : {baseline_acc:.3f}')
print(f'Lift over baseline   : {oos_acc - baseline_acc:+.3f}')
print()

print('Per-regime accuracy:')
for r in sorted(results_df['true_regime'].unique()):
    mask  = results_df['true_regime'] == r
    acc_r = accuracy_score(
        results_df.loc[mask, 'true_regime'],
        results_df.loc[mask, 'pred_regime']
    )
    label = REGIME_NAMES[int(r)] if int(r) < len(REGIME_NAMES) else f'Regime {int(r)}'
    print(f'  Regime {int(r)} ({label:<20s})  acc={acc_r:.3f}  n={int(mask.sum())}')

print()
print(classification_report(
    results_df['true_regime'],
    results_df['pred_regime'],
    target_names=[REGIME_NAMES[i] if i < len(REGIME_NAMES) else f'R{i}' for i in range(n_classes)]
))

---
## 7. Confusion Matrix

Two views of the same OOS classification results:

**Left panel (raw counts):** Each cell shows how many test days with true regime R were predicted as regime P. The diagonal = correct predictions. Off-diagonal = errors. Sum of any row = total test days for that true regime.

**Right panel (row-normalised / recall):** Each cell = fraction of true-regime-R days predicted as P. Reading a row: 'of all days that were actually regime R, what fraction did the model predict as each class?' The diagonal = per-class recall. This is more informative than raw counts when classes have different sizes.

**What to look for:**

| Error type | Acceptable? | Implication |
|-----------|-------------|-------------|
| Bull <-> Chop confusion | Yes | Adjacent regimes, similar conditions |
| Bear <-> Extreme Bear confusion | Yes | Same direction, different severity |
| Bull predicted during Bear | Concerning | Would cause strategy to enter in bear market |
| Bear/Extreme Bear predicted during Bull | Conservative | Strategy would miss some bull days, not catastrophic |

Conservative errors (false negatives on Bull) are preferable to aggressive errors (false positives on Bull). The model should be better at *avoiding* bad days than at *finding* good ones.

In [ ]:
labels_cm = [
    REGIME_NAMES[i] if i < len(REGIME_NAMES) else f'R{i}'
    for i in range(n_classes)
]
cm = confusion_matrix(results_df['true_regime'], results_df['pred_regime'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels_cm).plot(
    ax=axes[0], colorbar=True, cmap='Blues', values_format='d'
)
axes[0].set_title('OOS Confusion Matrix — counts\n(row = true, col = predicted)')

# Row-normalised (recall per class)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
ConfusionMatrixDisplay(confusion_matrix=cm_norm.round(2), display_labels=labels_cm).plot(
    ax=axes[1], colorbar=True, cmap='Blues', values_format='.2f'
)
axes[1].set_title('OOS Confusion Matrix — recall\n(fraction of true regime correctly predicted)')

plt.tight_layout()
plt.show()

---
## 8. Feature Importance

**Gain-based importance** measures the total reduction in the training loss when a feature is used as a split point, averaged across all trees in the ensemble and weighted by the number of samples affected. Higher gain = the feature makes more accurate splits.

This is averaged across all 16 walk-forward folds.

**Top 10 features (mean gain across folds):**

| Rank | Feature | Gain | Notes |
|------|---------|------|-------|
| 1 |  | 0.172 | 60-day drawdown — dominant signal by a wide margin |
| 2 |  | 0.094 | Overbought/oversold positioning |
| 3 |  | 0.077 | 60-day cumulative momentum |
| 4 |  | 0.062 | Short-term volatility level |
| 5 |  | 0.059 | 120-day drawdown (macro correction depth) |
| 6 |  | 0.047 | 1-year drawdown (full macro bear cycle depth) |
| 7 |  | 0.041 | Normalised daily range |
| 8 |  | 0.041 | Forward-filtered current regime |
| 9 |  | 0.035 | 20-day drawdown (near-term correction) |
| 10 |  | 0.035 | Price position within Bollinger bands |

**What the results show:**
-  consistently dominates at ~0.172 gain -- roughly double the next feature. The model's primary signal is simply how far price is from its 60-day peak: deep drawdown signals Bear/Extreme Bear, no drawdown signals Bull.
- The full drawdown family (, , , ) occupies ranks 1, 5, 6, and 9 -- four of the top ten features, covering every timescale from 3 weeks to a full year.
-  and  are the second-tier features alongside drawdown.
- The recovery features (, , ) do not appear in the top 10 -- they are highly correlated with the momentum features () and the gain is attributed to whichever correlated feature the tree splits on first. Despite low gain importance, they contribute to the improved backtest performance (Long/Flat Sharpe 0.69 vs 0.63 without them).

**What low importance does NOT mean:** Correlated features split importance between them. The recovery features are still used by the model -- their gain is diluted because  features capture overlapping information. The economic benefit shows in the backtest, not the gain ranking.

In [ ]:
# Mean gain importance across folds
importance_series = [
    pd.Series(
        dict(zip(m['feat_cols'], m['model'].feature_importances_)),
        name=f'fold_{m["fold"]}'
    )
    for m in all_meta
]
mean_imp = pd.concat(importance_series, axis=1).mean(axis=1).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 7))
mean_imp.head(20)[::-1].plot.barh(ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Mean feature importance (gain) — averaged across walk-forward folds\nTop 20 features')
ax.set_xlabel('Mean gain importance')
plt.tight_layout()
plt.show()

print('Top 10 features:')
for fname, imp in mean_imp.head(10).items():
    bar = chr(9608) * int(imp / mean_imp.iloc[0] * 30)
    print(f'  {fname:<25s}  {imp:.4f}  {bar}')

---
## 9. Hyperparameter Evolution

One of the key diagnostics of walk-forward: how do the Optuna-selected hyperparameters change across folds as the training window expands and market conditions change?

**What each parameter's evolution reveals:**

| Parameter | What drift tells you |
|-----------|---------------------|
| `n_estimators` | Trend upward as training set grows = more data allows more trees without overfitting |
| `max_depth` | Persistently high (6-7) = non-linear feature interactions are important; persistently low (3) = mostly linear |
| `learning_rate` | High variance fold-to-fold = the optimal learning rate is regime-dependent |
| `subsample` | Consistently low (<0.6) = the model benefits heavily from row bagging, suggesting overfitting risk |
| `reg_lambda` | High in some folds = L2 regularisation is being used aggressively; low in others = the data is sufficient for a less regularised model |
| `val_acc` | Trend downward over folds = newer market conditions are harder to predict |

**Boundary hitting:** If any parameter consistently lands at the edge of its search range (e.g. `reg_lambda` always = 0.1 or always = 5.0), the range should be widened in future runs. The current ranges were set specifically to avoid this issue.

In [ ]:
hp_history = pd.DataFrame([
    {
        'fold':             m['fold'],
        'test_start':       str(m['test_start']),
        'n_estimators':     m['best_params']['n_estimators'],
        'max_depth':        m['best_params']['max_depth'],
        'learning_rate':    m['best_params']['learning_rate'],
        'subsample':        m['best_params']['subsample'],
        'colsample_bytree': m['best_params']['colsample_bytree'],
        'gamma':            m['best_params']['gamma'],
        'reg_alpha':        m['best_params']['reg_alpha'],
        'reg_lambda':       m['best_params']['reg_lambda'],
        'val_acc':          m['val_acc'],
    }
    for m in all_meta
])

print('Hyperparameter summary across folds:')
print(hp_history.drop(columns=['fold', 'test_start']).describe().round(3).to_string())

hparams = ['n_estimators', 'max_depth', 'learning_rate',
           'subsample', 'colsample_bytree', 'gamma',
           'reg_alpha', 'reg_lambda', 'val_acc']

fig, axes = plt.subplots(3, 3, figsize=(14, 10))
for ax, hp in zip(axes.ravel(), hparams):
    ax.plot(hp_history['fold'], hp_history[hp], 'o-', color='steelblue', markersize=5)
    ax.set_title(hp, fontsize=10)
    ax.set_xlabel('Fold', fontsize=8)
    ax.tick_params(labelsize=8)
    if hp == 'learning_rate':
        ax.set_yscale('log')

plt.suptitle('Hyperparameter evolution across walk-forward folds', fontsize=12)
plt.tight_layout()
plt.show()

---
## 10. SHAP Analysis

SHAP (SHapley Additive exPlanations) provides a model-agnostic explanation of each prediction by decomposing it into the **additive contribution of each feature**. For a given day and class, the SHAP value of feature $j$ answers: 'how much did feature $j$ change the predicted probability of this regime, compared to the average prediction?'

Formally: $f(x) = \phi_0 + \sum_j \phi_j$, where $f(x)$ is the model output (log-odds), $\phi_0$ is the base rate, and $\phi_j$ is the SHAP value for feature $j$.

**Beeswarm plot -- how to read it:**
- Each horizontal strip = one feature (sorted by mean absolute SHAP, highest at top)
- Each dot = one test observation (one day)
- **X position** = SHAP value: right = pushed probability of this regime UP; left = pushed it DOWN
- **Colour** = feature value: red = high feature value; blue = low
- **Red dots on the right** = high feature value increases this regime's probability (positive relationship)
- **Blue dots on the right** = low feature value increases this regime's probability (inverse relationship)

**Example interpretation:** If `mom_60d` appears at the top of the Bull beeswarm with red dots on the right, it means: 'when 60-day momentum is high (red), the model assigns higher Bull probability (right). When it is low (blue), Bull probability drops.' This is economically intuitive -- strong recent returns predict continuation of the bull regime.

Three regime classes are shown (Bull, Chop, Bear). Extreme Bear is omitted to keep the output concise -- its SHAP structure mirrors Bear but with stronger magnitudes.

In [ ]:
# Use the last fold for SHAP
last_meta   = all_meta[-1]
model_shap  = last_meta['model']
scaler_shap = last_meta['scaler']
feat_shap   = last_meta['feat_cols']

# Reconstruct scaled test features for the last fold
last_idx  = results_df[results_df['fold'] == last_meta['fold']].index
last_raw  = feat.reindex(last_idx)[feat_shap].fillna(feat[feat_shap].median())
X_shap    = scaler_shap.transform(last_raw.values)
X_shap_df = pd.DataFrame(X_shap, columns=feat_shap)

explainer = shap.TreeExplainer(model_shap)
shap_vals = explainer.shap_values(X_shap)
# shap_vals shape: (n_samples, n_features, n_classes) in SHAP 0.42+
# extract per-class slice with [:, :, cls_idx]

n_show = min(n_classes, 3)
for cls_idx in range(n_show):
    label = REGIME_NAMES[cls_idx] if cls_idx < len(REGIME_NAMES) else f'Regime {cls_idx}'
    sv = shap_vals[:, :, cls_idx]
    print(f'SHAP beeswarm — class {cls_idx}: {label}')
    shap.summary_plot(
        sv,
        X_shap_df,
        feature_names=feat_shap,
        max_display=12,
        show=False,
        plot_type='dot',
    )
    plt.title(f'SHAP — {label} regime (class {cls_idx})')
    plt.tight_layout()
    plt.show()

---
## 11. SHAP Dependence Plots

Dependence plots show the **exact non-linear relationship** between a single feature's value and its SHAP contribution to a specific regime class.

- **X-axis**: raw feature value (on the original scale, before normalisation)
- **Y-axis**: SHAP value for this feature (positive = pushes this regime's probability up)
- **Point colour**: the most important *interaction* feature -- the feature whose value explains the residual scatter in the SHAP values after accounting for the main effect

**What this reveals that importance plots hide:**
- Threshold effects: the SHAP contribution may only become significant above or below a certain value
- Asymmetry: the effect of high feature values may be much stronger than low values (or vice versa)
- Interactions: coloured scatter that groups by value of the interaction feature shows the model has learned a joint effect

The top feature for Bull and the top feature for Bear are shown. These are the variables the model relies on most when making high-confidence regime calls in each direction.

In [ ]:
# SHAP dependence for top feature in Bull regime (class 0)
bull_shap    = shap_vals[:, :, 0]
top_feat_idx = int(np.abs(bull_shap).mean(0).argmax())
top_feat     = feat_shap[top_feat_idx]

print(f'Top driver for Bull regime prediction: {top_feat}')
shap.dependence_plot(
    top_feat_idx,
    bull_shap,
    X_shap_df,
    feature_names=feat_shap,
    show=False,
)
plt.title(f'SHAP dependence — {top_feat}  (Bull regime, last fold)')
plt.tight_layout()
plt.show()

if n_classes >= 3:
    bear_shap     = shap_vals[:, :, 2]
    top_bear_idx  = int(np.abs(bear_shap).mean(0).argmax())
    top_bear_feat = feat_shap[top_bear_idx]
    print(f'Top driver for Bear regime prediction: {top_bear_feat}')
    shap.dependence_plot(
        top_bear_idx,
        bear_shap,
        X_shap_df,
        feature_names=feat_shap,
        show=False,
    )
    plt.title(f'SHAP dependence — {top_bear_feat}  (Bear regime, last fold)')
    plt.tight_layout()
    plt.show()

---
## 12. Predicted vs True Regime Timeline

Visual OOS validation across the full 1,620-day test period. Three panels:
- **Top**: BTC price (log scale) shaded by the *true* HMM Viterbi regime
- **Middle strip**: True regime colour, day by day
- **Bottom strip**: Predicted regime colour, day by day

**How to read it:**
- Identical top and bottom strips = perfect prediction
- 1-2 day lag at boundaries = acceptable; the model cannot anticipate a regime it has not seen start yet
- Short incorrect patches (2-5 days) at transitions = normal; the Viterbi true label may already have the new regime while features have not yet changed enough for XGBoost to detect it
- Sustained wrong-regime periods (>2 weeks) = a genuine failure mode worth investigating -- often corresponds to low val_acc folds

Pay attention to the 2022 bear market: this is the hardest test. If the model correctly shows Bear/Extreme Bear labels from early 2022 through late 2022, the regime filter would have protected any directional strategy from that entire period.

In [ ]:
price_oos = close.reindex(results_df.index)

fig = plt.figure(figsize=(16, 9))
gs  = gridspec.GridSpec(3, 1, height_ratios=[5, 1, 1], hspace=0.04)
ax0 = fig.add_subplot(gs[0])
ax1 = fig.add_subplot(gs[1], sharex=ax0)
ax2 = fig.add_subplot(gs[2], sharex=ax0)

# Price with true regime shading
ax0.semilogy(price_oos.index, price_oos.values, color='black', linewidth=0.9, zorder=3)

prev_r, prev_date = int(results_df['true_regime'].iloc[0]), results_df.index[0]
for i in range(1, len(results_df)):
    r = int(results_df['true_regime'].iloc[i])
    if r != prev_r or i == len(results_df) - 1:
        ax0.axvspan(prev_date, results_df.index[i],
                    color=REGIME_COLORS[prev_r], alpha=0.2, zorder=1)
        prev_r, prev_date = r, results_df.index[i]

from matplotlib.patches import Patch
legend_els = [
    Patch(color=REGIME_COLORS[r], alpha=0.55,
          label=REGIME_NAMES[r] if r < len(REGIME_NAMES) else f'R{r}')
    for r in range(n_classes)
]
ax0.legend(handles=legend_els, fontsize=8, loc='upper left')
ax0.set_ylabel('BTC price (log scale)')
ax0.set_title('BTC price — true HMM regime shading (OOS period)')
plt.setp(ax0.get_xticklabels(), visible=False)

# True regime colour strip
ax1.bar(
    results_df.index, np.ones(len(results_df)),
    color=[REGIME_COLORS[int(r)] for r in results_df['true_regime']],
    width=1.5, align='edge', linewidth=0,
)
ax1.set_ylabel('True', fontsize=9)
ax1.set_ylim(0, 1)
ax1.set_yticks([])
plt.setp(ax1.get_xticklabels(), visible=False)

# Predicted regime colour strip
ax2.bar(
    results_df.index, np.ones(len(results_df)),
    color=[REGIME_COLORS[int(r)] for r in results_df['pred_regime']],
    width=1.5, align='edge', linewidth=0,
)
ax2.set_ylabel('Pred', fontsize=9)
ax2.set_ylim(0, 1)
ax2.set_yticks([])

plt.suptitle('True (HMM) vs Predicted (XGBoost) regimes — OOS', y=1.01)
plt.tight_layout()
plt.show()

---
## 13. Regime Probability Time Series

Rather than just a hard label, the model outputs a **probability distribution** over all four regimes for each day. These probabilities are what make probability-threshold filtering possible.

**Stacked area chart (top):**
- Wide single-colour bands = high model confidence (one regime dominates, probability near 1.0)
- Mixed narrow bands = uncertainty at transitions -- the model is 50/50 between two adjacent regimes
- The uncertain sections are exactly where a probability threshold filter adds value: a hard label would say 'Bull', but p_regime_0 = 0.52 is not the same confidence level as p_regime_0 = 0.91

**Bull probability panel (bottom):**
p_regime_0 shown alone against price. Key questions to check:
- Does p(Bull) rise *before* large price advances? That would mean the model has genuine predictive power, not just contemporaneous correlation
- Does p(Bull) stay low during the 2022 bear market? It should -- otherwise the model would incorrectly allow long positions throughout
- Are there periods where p(Bull) is high but price goes sideways? Those are Chop periods the model mislabels as Bull -- exactly the entries a threshold of 0.70 would filter out

**Using probabilities in strategies:** See `topics/momentum/results/moneyin_regime.ipynb` for a full analysis of probability thresholding vs hard regime filtering on the MoneyIn Long strategy. The recommended configuration is Hybrid 0.70: allow positions when p_regime_0 > 0.70 OR predicted regime == 1 (Chop always allowed).

In [ ]:
prob_cols = [f'p_regime_{i}' for i in range(n_classes)]

fig, ax = plt.subplots(figsize=(16, 4))
bottom = np.zeros(len(results_df))
for i, col in enumerate(prob_cols):
    label = REGIME_NAMES[i] if i < len(REGIME_NAMES) else f'Regime {i}'
    ax.fill_between(
        results_df.index, bottom, bottom + results_df[col].values,
        color=REGIME_COLORS[i], alpha=0.75, label=label
    )
    bottom += results_df[col].values

ax.set_ylim(0, 1)
ax.set_ylabel('Predicted regime probability')
ax.set_title('XGBoost regime probabilities — OOS period (stacked)')
ax.legend(fontsize=8, loc='lower left')
plt.tight_layout()
plt.show()

# Bull regime probability alone
fig, axes = plt.subplots(2, 1, figsize=(16, 6), sharex=True)
axes[0].semilogy(price_oos.index, price_oos.values, color='black', linewidth=0.8)
axes[0].set_ylabel('BTC price')
axes[0].set_title('BTC price and predicted Bull regime probability')
axes[1].fill_between(results_df.index, 0, results_df['p_regime_0'],
                     color=REGIME_COLORS[0], alpha=0.6)
axes[1].axhline(0.5, color='black', linewidth=0.7, linestyle='--', label='p=0.5')
axes[1].set_ylabel('P(Bull)')
axes[1].set_ylim(0, 1)
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

---
## 14. Save OOS Predictions

The OOS predictions parquet is the primary output of this notebook. It contains one row per test day, across all 18 folds, with:
- `p_regime_0` through `p_regime_3`: XGBoost-predicted probability that *tomorrow* is each regime
- `pred_regime`: argmax of the four probabilities -- the hard predicted label
- `true_regime`: the actual Viterbi regime (ground truth, for evaluation only)
- `fold`: which walk-forward fold produced this prediction
- `val_acc`: the Optuna validation accuracy for that fold

**Important note on `p_regime_0`:** In `btc_regime_predictions.parquet`, `p_regime_0` is the XGBoost probability that **tomorrow** is Bull. In `btc_regimes.parquet` from notebook 1, `p_regime_0` is the HMM forward-backward probability that **today** is Bull. These are different quantities -- do not mix them.

In [ ]:
out_path = PREDS_DIR / 'btc_regime_predictions.parquet'
results_df.to_parquet(out_path, engine='pyarrow')

print(f'Saved: {out_path}')
print(f'{len(results_df)} rows  |  {len(all_meta)} folds')
print(f'{results_df.index[0].date()} to {results_df.index[-1].date()}')
print(f'Columns: {results_df.columns.tolist()}')

# Fold summary table
summary = pd.DataFrame([
    {
        'fold':       m['fold'],
        'test_start': m['test_start'],
        'test_end':   m['test_end'],
        'val_acc':    round(m['val_acc'], 3),
        'n_est':      m['best_params']['n_estimators'],
        'depth':      m['best_params']['max_depth'],
        'lr':         round(m['best_params']['learning_rate'], 4),
        'subsample':  round(m['best_params']['subsample'], 3),
    }
    for m in all_meta
])
print()
print('Fold summary:')
print(summary.to_string(index=False))

---
## 15. Direct Regime-as-Strategy Backtest

A simple but powerful economic validation: use the OOS regime predictions as a direct BTC trading signal. No additional strategy logic -- just map predicted regime to position.

**Three variants:**

| Strategy | Bull | Chop | Bear | Extreme Bear | Rationale |
|----------|------|------|------|--------------|----------|
| Long / Flat / Short | +1 | 0 | -1 | -1 | Full regime signal |
| **Long / Flat only** | **+1** | **0** | **0** | **0** | Conservative -- avoid short squeezes |
| Long / Half / Flat | +1 | +0.5 | 0 | 0 | Partial exposure during Chop |

Transaction cost: 6 bps applied at every position change.

**What to look for:**
- Long/Flat should outperform Buy & Hold on a risk-adjusted basis (Sharpe, MaxDD) even if total return is lower -- because it avoids the worst bear market periods
- Long/Flat/Short is a stronger signal but carries short risk (short squeezes); its Sharpe should be higher than Long/Flat if the Bear predictions are accurate
- The **forward return table** at the bottom of the cell is the key sanity check: average annualised return following each predicted regime should be monotonically ordered (Bull highest, Extreme Bear lowest). If not, the model's predicted labels do not have genuine directional predictive power.

**Interpreting the results:** This backtest uses the regime predictions directly -- it is not a traded strategy. The real value of the regime filter is in improving other strategies (see `topics/momentum/results/moneyin_regime.ipynb`). This section serves as an independent sanity check that the labels contain economic signal.

In [ ]:
COST_BPS = 6

btc_oos = np.log(close.reindex(results_df.index) / close.reindex(results_df.index).shift(1))

def backtest(pos_map, label):
    pos  = results_df['pred_regime'].map(pos_map)
    ret  = pos.shift(1) * btc_oos
    cost = (pos != pos.shift(1)).astype(float) * (COST_BPS / 10_000)
    ret  = (ret - cost).dropna()
    eq   = (1 + ret).cumprod()
    sh   = ret.mean() / ret.std() * np.sqrt(365) if ret.std() > 0 else 0
    mdd  = float((eq / eq.cummax() - 1).min())
    ann  = float(np.exp(ret.sum() * 365 / len(ret)) - 1)
    tot  = float(eq.iloc[-1] - 1)
    return {'label': label, 'ret': ret, 'eq': eq, 'sharpe': sh,
            'mdd': mdd, 'ann': ann, 'total': tot,
            'in_mkt': int((pos.shift(1) != 0).sum()),
            'trades': int((pos != pos.shift(1)).sum())}

strategies = [
    backtest({0:1, 1:0, 2:-1, 3:-1}, 'Long / Flat / Short'),
    backtest({0:1, 1:0, 2: 0, 3: 0}, 'Long / Flat only'),
    backtest({0:1, 1:0, 2: 0, 3:-1}, 'Long / Flat / ExBear Short'),
    backtest({0:1, 1:0.5,2: 0, 3: 0}, 'Long / Half / Flat'),
]

bnh_ret = btc_oos.reindex(strategies[0]['ret'].index)
bnh_eq  = (1 + bnh_ret).cumprod()
bnh = {'label': 'Buy & Hold', 'ret': bnh_ret, 'eq': bnh_eq,
       'sharpe': bnh_ret.mean()/bnh_ret.std()*np.sqrt(365),
       'mdd': float((bnh_eq/bnh_eq.cummax()-1).min()),
       'ann': float(np.exp(bnh_ret.sum()*365/len(bnh_ret))-1),
       'total': float(bnh_eq.iloc[-1]-1)}

print(f'Period: {strategies[0]["ret"].index[0].date()} to {strategies[0]["ret"].index[-1].date()}')
print()
print(f'  {"Strategy":<30s}  {"Ann Ret":>8s}  {"Sharpe":>7s}  {"Max DD":>8s}  {"Total":>8s}  {"In Mkt":>7s}')
print('  ' + '-'*74)
for s in strategies:
    print(f'  {s["label"]:<30s}  {s["ann"]:>8.1%}  {s["sharpe"]:>7.2f}  {s["mdd"]:>8.1%}  {s["total"]:>8.1%}  {s["in_mkt"]:>6d}d')
print(f'  {bnh["label"]:<30s}  {bnh["ann"]:>8.1%}  {bnh["sharpe"]:>7.2f}  {bnh["mdd"]:>8.1%}  {bnh["total"]:>8.1%}')

print()
print('Avg annualised return on days following each predicted regime:')
fwd_ret = btc_oos.reindex(results_df.index).shift(-1)
for r, name in enumerate(REGIME_NAMES[:n_classes]):
    mask = results_df['pred_regime'] == r
    avg  = fwd_ret[mask].mean() * 365 * 100
    print(f'  {name:<20s}  {avg:>+7.1f}% ann  n={int(mask.sum())}')


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(15, 11), sharex=True,
                         gridspec_kw={'height_ratios': [4, 2, 1.5]})

colors = {'Long / Flat / Short':       '#e53935',
          'Long / Flat only':          '#1565c0',
          'Long / Flat / ExBear Short':'#6a1b9a',
          'Long / Half / Flat':        '#2e7d32'}

for s in strategies:
    axes[0].plot(s['eq'].index, s['eq'].values, label=s['label'],
                 color=colors[s['label']], linewidth=1.4)
axes[0].plot(bnh['eq'].index, bnh['eq'].values, label='Buy & Hold',
             color='grey', linewidth=1.0, alpha=0.6, linestyle='--')
axes[0].set_ylabel('Cumulative return')
axes[0].set_title('Regime strategy variants vs Buy-and-Hold — OOS period')
axes[0].legend(fontsize=9)
axes[0].axhline(1, color='black', linewidth=0.4, linestyle='--')
plt.setp(axes[0].get_xticklabels(), visible=False)

best = strategies[1]
dd_s = (best['eq'] / best['eq'].cummax() - 1) * 100
dd_b = (bnh['eq']  / bnh['eq'].cummax()  - 1) * 100
axes[1].fill_between(dd_s.index, dd_s.values, 0, color='#1565c0', alpha=0.45, label='Long/Flat DD')
axes[1].fill_between(dd_b.index, dd_b.values, 0, color='grey',    alpha=0.30, label='B&H DD')
axes[1].set_ylabel('Drawdown %')
axes[1].legend(fontsize=8)
plt.setp(axes[1].get_xticklabels(), visible=False)

axes[2].bar(
    results_df.index, np.ones(len(results_df)),
    color=[REGIME_COLORS[int(r)] if int(r) < len(REGIME_COLORS) else 'grey'
           for r in results_df['pred_regime']],
    width=1.5, align='edge', linewidth=0
)
axes[2].set_ylabel('Predicted regime')
axes[2].set_yticks([])

plt.tight_layout()
plt.show()

